# 🔴 Day 10 — Structured Output & Workflow Patterns

**LangGraph from Zero to Advanced · Day 10 of 16**

**Today's goal:** get **typed data** (not just text) from the LLM, and learn the **workflow patterns** used in almost every real LLM app.

> **Workflow vs Agent:** in a *workflow* **you** decide the steps (fixed graph). In an *agent* the **LLM** decides (Day 6). Workflows are cheaper, faster and more predictable — use them whenever you can.

| Lesson | Pattern |
|---|---|
| 1 | Structured output with Pydantic |
| 2 | Prompt chaining (with a quality gate) |
| 3 | Routing (LLM picks the path) |
| 4 | Parallelization (fan-out → combine) |
| 5 | Evaluator–optimizer (write → grade → improve loop) |
| 6 | Practice |

(The 5th classic pattern, **orchestrator–workers**, is Day 11.)

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — Structured output

Parsing free text is fragile. `llm.with_structured_output(MyModel)` makes the LLM return a **Pydantic object** with exactly the fields you define. The `Field(description=...)` text guides the LLM.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Review(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="overall feeling")
    rating: int = Field(description="star rating from 1 to 5")
    key_points: list[str] = Field(description="main points, max 3, very short")

review_llm = llm.with_structured_output(Review)

r = review_llm.invoke("Food at this Pune restaurant was amazing and cheap, but the service was slow.")
print(type(r).__name__)
print("sentiment :", r.sentiment)
print("rating    :", r.rating)
print("key_points:", r.key_points)

Now you can use normal Python: `if r.rating < 3: alert_manager()`. This is how LLM output drives **routing decisions** in graphs.

## Lesson 2 — Prompt chaining (with a gate)

Break a task into steps; each LLM call works on the previous output. Add a plain-Python **gate** to check quality.

```
START ► write_tagline ► check ─ ok ──────────────► END
                          └─ too long ► shorten ► END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class TaglineState(TypedDict):
    product: str
    tagline: str

def write_tagline(state):
    return {"tagline": llm.invoke(f"Write a catchy tagline for: {state['product']}. Reply with the tagline only.").content}

def check(state) -> Literal["shorten", "__end__"]:
    return "shorten" if len(state["tagline"].split()) > 6 else END      # plain Python gate

def shorten(state):
    return {"tagline": llm.invoke(f"Shorten to max 6 words, keep it catchy: {state['tagline']}").content}

g = StateGraph(TaglineState)
g.add_node("write_tagline", write_tagline)
g.add_node("shorten", shorten)
g.add_edge(START, "write_tagline")
g.add_conditional_edges("write_tagline", check)
g.add_edge("shorten", END)
chain_app = g.compile()
show_graph(chain_app)
print(chain_app.invoke({"product": "an app that reminds you to drink water"})["tagline"])

## Lesson 3 — Routing (LLM picks the path)

On Day 3 we routed with keywords. Now an **LLM classifier** with structured output chooses — much smarter.

```
              ┌► billing_agent ─┐
START ► route ┼► tech_agent ────┼► END
              └► general_agent ─┘
```

In [ ]:
class Route(BaseModel):
    team: Literal["billing", "tech", "general"] = Field(description="which team should handle the message")

router_llm = llm.with_structured_output(Route)

class SupportState(TypedDict):
    message: str
    team: str
    reply: str

def route(state):
    decision = router_llm.invoke(f"Classify this customer message: {state['message']}")
    return {"team": decision.team}

def make_team(team, style):
    def node(state):
        reply = llm.invoke(f"You are the {team} support team. {style} Reply in 2 sentences to: {state['message']}")
        return {"reply": reply.content}
    return node

g = StateGraph(SupportState)
g.add_node("route", route)
g.add_node("billing", make_team("billing", "Be precise about money."))
g.add_node("tech", make_team("tech", "Give one clear troubleshooting step."))
g.add_node("general", make_team("general", "Be warm and friendly."))
g.add_edge(START, "route")
g.add_conditional_edges("route", lambda s: s["team"], ["billing", "tech", "general"])
for t in ("billing", "tech", "general"):
    g.add_edge(t, END)
support = g.compile()
show_graph(support)

for msg in ["I was charged twice for my order", "The app freezes on the login screen", "Do you have a store in Indore?"]:
    out = support.invoke({"message": msg})
    print(f"👤 {msg}\n   → {out['team']}: {out['reply']}\n")

## Lesson 4 — Parallelization

Run independent LLM calls **at the same time**, then combine. Faster, and each call stays focused.

```
          ┌► pros ──┐
START ────┼► cons ──┼► combine ► END
          └► risks ─┘
```

In [ ]:
class DecisionState(TypedDict):
    question: str
    pros: str
    cons: str
    risks: str
    final: str

def pros(state):  return {"pros":  llm.invoke(f"List 3 short pros of: {state['question']}").content}
def cons(state):  return {"cons":  llm.invoke(f"List 3 short cons of: {state['question']}").content}
def risks(state): return {"risks": llm.invoke(f"List 2 short hidden risks of: {state['question']}").content}

def combine(state):
    prompt = (f"Question: {state['question']}\nPros: {state['pros']}\nCons: {state['cons']}\n"
              f"Risks: {state['risks']}\nGive a balanced 2-sentence recommendation.")
    return {"final": llm.invoke(prompt).content}

g = StateGraph(DecisionState)
for fn in (pros, cons, risks, combine):
    g.add_node(fn.__name__, fn)
for fn in ("pros", "cons", "risks"):
    g.add_edge(START, fn)
g.add_edge(["pros", "cons", "risks"], "combine")     # 👈 wait for ALL three
g.add_edge("combine", END)
decide = g.compile()
show_graph(decide)

print(decide.invoke({"question": "Should a fresher learn LangGraph before getting a job?"})["final"])

💡 `add_edge(["a", "b", "c"], "combine")` = *run `combine` only after **all** of a, b, c finish.* Each parallel node writes a **different** key, so no reducer is needed here.

## Lesson 5 — Evaluator–optimizer

One LLM **writes**, another **grades** with structured output and gives feedback. Loop until it passes (or max attempts).

```
START ► write ► evaluate ─ pass / 3 tries ──► END
          ▲          │
          └─ fail ───┘   (with feedback)
```

In [ ]:
class Grade(BaseModel):
    grade: Literal["pass", "fail"] = Field(description="pass only if all rules are followed")
    feedback: str = Field(description="what to fix if failed, else empty")

grader = llm.with_structured_output(Grade)

class PoemState(TypedDict):
    topic: str
    poem: str
    grade: str
    feedback: str
    attempts: int

RULES = "Exactly 4 lines, must rhyme, must mention the word 'chai'."

def write(state):
    prompt = f"Write a poem about {state['topic']}. Rules: {RULES}"
    if state.get("feedback"):
        prompt += f"\nYour previous poem:\n{state['poem']}\nFix this feedback: {state['feedback']}"
    return {"poem": llm.invoke(prompt).content, "attempts": state.get("attempts", 0) + 1}

def evaluate(state):
    result = grader.invoke(f"Rules: {RULES}\nGrade this poem strictly:\n{state['poem']}")
    return {"grade": result.grade, "feedback": result.feedback}

def good_enough(state) -> Literal["write", "__end__"]:
    if state["grade"] == "pass" or state["attempts"] >= 3:
        return END
    return "write"

g = StateGraph(PoemState)
g.add_node("write", write)
g.add_node("evaluate", evaluate)
g.add_edge(START, "write")
g.add_edge("write", "evaluate")
g.add_conditional_edges("evaluate", good_enough)
poet = g.compile()
show_graph(poet)

out = poet.invoke({"topic": "rainy evenings"})
print(f"attempts={out['attempts']} | grade={out['grade']}\n")
print(out["poem"])

## Lesson 6 — Practice

**Exercise 1.** Extract a structured `Invoice` (`vendor: str`, `amount: float`, `currency: str`, `due_date: str`) from:
*"Hi, please pay ₹12,500 to Sharma Electricals by 15 Nov 2026."*

**Exercise 2.** Build a parallel graph that translates one sentence into **Hindi, Tamil and Bengali** at the same time, then a `combine` node prints all three.

In [ ]:
# ✅ Solution 1
class Invoice(BaseModel):
    vendor: str
    amount: float
    currency: str = Field(description="3-letter code, e.g. INR")
    due_date: str = Field(description="YYYY-MM-DD")

inv = llm.with_structured_output(Invoice).invoke("Hi, please pay ₹12,500 to Sharma Electricals by 15 Nov 2026.")
print(inv)

In [ ]:
# ✅ Solution 2
class TransState(TypedDict):
    text: str
    hindi: str
    tamil: str
    bengali: str
    all: str

def make_translator(lang):
    def node(state):
        return {lang: llm.invoke(f"Translate into {lang.title()}, reply with translation only: {state['text']}").content}
    return node

g = StateGraph(TransState)
for lang in ("hindi", "tamil", "bengali"):
    g.add_node(lang, make_translator(lang))
    g.add_edge(START, lang)
g.add_node("combine", lambda s: {"all": f"HI: {s['hindi']}\nTA: {s['tamil']}\nBN: {s['bengali']}"})
g.add_edge(["hindi", "tamil", "bengali"], "combine")
g.add_edge("combine", END)
print(g.compile().invoke({"text": "Practice every day."})["all"])
print("✅ All exercises solved!")

## 🎯 Day 10 Recap

* `llm.with_structured_output(PydanticModel)` → reliable typed data.
* **Chaining:** step by step, with code gates.
* **Routing:** LLM classifier picks the branch.
* **Parallelization:** fan-out, then `add_edge([...], "combine")`.
* **Evaluator–optimizer:** write → grade → improve, with a max-attempts limit.

**Tomorrow (Day 11):** dynamic parallel work with **`Send`** (map-reduce) — the orchestrator–workers pattern.